# Silver Layer__CRM Product
### CRM_PRD_INFO --> CRM_PRODUCTS
Cleans and transforms the bronze CRM product data and loads it into operation.silver.crm_products.



### Section 1 — Read data from the Bronze table into a DataFrame

In [0]:
SOURCE_TABLE='operation.bronze.crm_prd_info'
TARGET_TABLE='operation.silver.crm_products'

# read data from bronze
from pyspark.sql import functions as F, window as W
df_bronze=spark.table(SOURCE_TABLE)
print(f'Load table {SOURCE_TABLE} from bronze layer completed with ',df_bronze.count(),' rows.','\n')
df_bronze.printSchema()
df_bronze.show(5)

### Section 2 — Analyze data quality using SQL (list all identified issues)
- Found 0 duplicate **prd_id**
- Found 77 duplicate **prd_key**
- Nulls per columns:
    prd_id=0 ,
    prd_key=0 ,
    prd_nm=0 ,
    prd_cost=2 ,
    prd_line=17 ,
    prd_start_dt=0 ,
    prd_end_dt=197 

- Whitespace in string columns:

        Found 0 rows with leading or trailing whitespace in column **prd_id**

        Found 0 rows with leading or trailing whitespace in column **prd_key**

        Found 0 rows with leading or trailing whitespace in column **prd_nm**

        Found 0 rows with leading or trailing whitespace in column **prd_cost**

        Found 380 rows with leading or trailing whitespace in column **prd_line**
- abbrivation **prd_line**:
        
         R : "Road", S : "Other Sales", M : "Mountain", T : "Touring" 
- number of rows for column **prd_start_dt** that string unparseable as yyyy-MM-dd :  0
- number of rows for column **prd_end_dt** that string unparseable as yyyy-MM-dd :  0


        start_date ( min: `2003-07-01`, max: `2013-07-01`)
        end_date (min: `2007-12-28`, max: `2008-12-27`)

        
       

In [0]:
# DQ check 1 — duplicates on the business key (prd_id) and the surrogate key (prd_key), shown separately
dup_prd_id=df_bronze.groupby('prd_id').count().filter(F.col('count')>1)
print(f'Found {dup_prd_id.count()} duplicate prd_id')

dup_prd_key=df_bronze.groupby('prd_key').count().filter(F.col('count')>1)
print(f'Found{dup_prd_key.count()} duplicate prd_key')
dup_prd_key.show()
df_bronze.join(dup_prd_key,on="prd_key",how='inner').show()


In [0]:
# DQ check 2 — nulls per column
print('_____DQ check 2 — nulls per column_______')
print('\n')
null_counts=df_bronze.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df_bronze.columns])
null_counts.show()

# DQ check 3 — leading/trailing whitespace in string columns
print('\n')
print('______DQ check 3 — leading/trailing whitespace in string columns________')
print('\n')
for c in ['prd_id','prd_key','prd_nm','prd_cost','prd_line']:
    n=df_bronze.filter(F.col(c)!=F.trim(F.col(c))).count()
    print(f'Found {n} rows with leading or trailing whitespace in column {c}')
# DQ check 4 — distinct categorical values (what needs normalizing?)
print('\n')
print('__________DQ check 4 — distinct categorical values________')
print('\n')
df_bronze.select(F.col('prd_line')).distinct().show()

# DQ check 5 — date format & range: is the string parseable as yyyy-MM-dd, and is the range plausible?
print('\n')
print('______DQ check 5 — date format & range________')
print('\n')
df_bronze.select(F.max(F.col('prd_start_dt')).alias('max_start_date'),F.min(F.col('prd_start_dt')).alias('min_start_date'),F.min(F.col('prd_end_dt')).alias('min_end_date'),F.max(F.col('prd_end_dt')).alias('max_end_date')).show()

print('\n')
for c in ['prd_start_dt','prd_end_dt']:
    n=df_bronze.filter( F.col(c).isNotNull() & F.to_date(F.col(c),'yyyy-MM-dd').isNull())
    print(f'''number of rows for column {c} that string unparseable as yyyy-MM-dd : ''',n.count())
print('\n')

### Section 2.1 — Transform data (fix issues one by one)

In [0]:
df_clean=(df_bronze
          .withColumn('prd_id' , F.trim(F.col('prd_id')).cast('int') )
          .withColumn('prd_key', F.trim(F.col('prd_key')))
          .withColumn('prd_nm', F.trim(F.col('prd_nm')))
          .withColumn('prd_cost', F.trim(F.col('prd_cost')).cast('int'))
          .withColumn('prd_line', F.when(F.trim(F.col('prd_line'))=='M','Mountain')
                     .when(F.trim(F.col('prd_line'))=='R','Road')
                     .when(F.trim(F.col('prd_line'))=='S','Other Sales')
                     .when(F.trim(F.col('prd_line'))=='T','Touring')
                     .otherwise("n/a"))
          .withColumn('prd_start_dt', F.to_date(F.trim(F.col('prd_start_dt')),'yyyy-MM-dd'))
          .withColumn('prd_end_dt', F.to_date(F.trim(F.col('prd_end_dt')),'yyyy-MM-dd'))
          )
df_clean=df_clean.select(
            F.col('prd_id').alias('Product_id'),
            F.col('prd_key').alias('Product_key'),
            F.col('prd_nm').alias('Product_name'),        
            F.col('prd_cost').alias('Product_cost'),
            F.col('prd_line').alias('Product_line'),
            F.col('prd_start_dt').alias('Product_start_date'),
            F.col('prd_end_dt').alias('Product_end_date'))
# check
df_clean.printSchema()

display(df_clean.groupBy(F.col('Product_line')).count())

### Section 3 — Write the DataFrame to a new Silver table (friendly name)

In [0]:
df_clean.write.mode('overwrite').saveAsTable(TARGET_TABLE)
print(f'Write table {TARGET_TABLE} to silver layer completed with ',df_clean.count(),' rows.')
df_silver=spark.table(TARGET_TABLE)
df_silver.printSchema()
df_silver.show(5)
